# LAPD Crime Data 2020–2024 — EDA

Group project for ISM Data Science 2026. Source: [catalog.data.gov](https://catalog.data.gov/dataset/crime-data-from-2020-to-present), dataset id `2nrs-mtv8`, sourced from `data.lacity.org`.

## What's in the dataset

Just over **a million reported crime incidents in Los Angeles** between January 2020 and December 2024. Each row is one incident: when it happened, where (lat/lon + LAPD reporting district), what crime code(s) were assigned, the victim's age/sex/descent if known, the premises type, weapon (if any), and case status.

LAPD migrated to NIBRS in March 2024, so this CSV is **frozen** — it's the official "historical reference only" snapshot. Good for us: nothing about it will change between now and our presentation.

## Why we picked it

It's rich on every dimension that EDA cares about — geo, time, taxonomy, and demographics — so we have lots of room to chase down interesting questions instead of forcing a single hypothesis.

## 1. Loading the data

I'm using `duckdb` for the initial load because it's faster than `pandas.read_csv` on this file size and auto-parses dates correctly. After the SQL pull I move to pandas for the rest because we want the full `groupby` / plotting stack.

In [ ]:
import duckdb
import pandas as pd
import numpy as np

DATA_PATH = "/Users/mac/Downloads/datamining-main/data/data.csv"

df = duckdb.sql(f"""
    SELECT * FROM read_csv_auto('{DATA_PATH}', header=TRUE)
""").df()

print(df.shape)
df.head(5)

In [ ]:
df.dtypes

A million rows, 28 columns. The columns I'll lean on are `DATE OCC`, `TIME OCC`, `AREA NAME`, `Crm Cd`, `Crm Cd Desc`, `Vict Age/Sex/Descent`, `Premis Desc`, `LAT`, `LON`. Most of the rest is administrative.

## 2. Preprocessing

### 2.1 Missing values

Quick scan of what's missing and how much.

In [ ]:
df.isna().sum().sort_values(ascending=False).head(15)

The picture:
- `Crm Cd 2/3/4` are secondary crime codes — empty for >99% of incidents (most crimes have only one code). I'll drop these.
- `Mocodes` is a free-text MO-code string — we won't analyze it, drop.
- `Weapon Used Cd` and `Weapon Desc` are missing when no weapon was used. I'll keep `Weapon Desc` and fill NaN with `"None"`.
- `Vict Sex`, `Vict Descent`, `Premis Desc` are sparsely missing — fill with `"Unknown"`.

In [ ]:
df = df.drop(columns=[
    "Crm Cd 1", "Crm Cd 2", "Crm Cd 3", "Crm Cd 4",
    "Mocodes", "Weapon Used Cd",
])

df["Weapon Desc"]  = df["Weapon Desc"].fillna("None")
df["Vict Sex"]     = df["Vict Sex"].fillna("Unknown")
df["Vict Descent"] = df["Vict Descent"].fillna("Unknown")
df["Premis Desc"]  = df["Premis Desc"].fillna("Unknown")

df = df.dropna(subset=["Premis Cd", "Status"])

print(df.shape)

### 2.2 Column names

LAPD's headers are inconsistent — `DR_NO`, `Date Rptd`, `AREA NAME`, `Crm Cd`. I'll normalize everything to `snake_case` so the rest of the notebook is quoting-free.

In [ ]:
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
)
df.columns.tolist()

### 2.3 Dates and time

`date_rptd` and `date_occ` come in as `datetime64` already from duckdb. `time_occ` is the 4-digit military time as an integer like `845` or `2030`. I'll zero-pad to a 4-char string and pull out the hour for time-of-day analysis.

In [ ]:
df["date_occ"]  = pd.to_datetime(df["date_occ"])
df["date_rptd"] = pd.to_datetime(df["date_rptd"])

df["time_occ"] = df["time_occ"].astype(str).str.zfill(4)
df["hour"]     = df["time_occ"].str[:2].astype(int)

df[["date_occ", "time_occ", "hour"]].head()

### 2.4 Coordinates

Some rows have `(0, 0)` as the location — that's a placeholder, not a real fix. Anything outside LA's bounding box (`lat 33-35`, `lon -119 to -118`) is also suspect.

In [ ]:
bad_coords = ((df["lat"] == 0) & (df["lon"] == 0)).sum()
print(f"Rows with (0, 0) coordinates: {bad_coords:,}")

df = df[(df["lat"] != 0) & (df["lon"] != 0)]
df = df[(df["lat"].between(33, 35)) & (df["lon"].between(-119, -118))]
print(f"Rows after coord filter: {len(df):,}")

### 2.5 Victim age

Some victim ages are negative or wildly large — likely sentinel values for "unknown". I keep `0 <= age <= 100`. Note: `age == 0` here means "no victim age recorded" (often property crimes), not "infant". I keep them because dropping them would skew the area/time analysis.

In [ ]:
print("Before:", df["vict_age"].min(), df["vict_age"].max())
df = df[(df["vict_age"] >= 0) & (df["vict_age"] <= 100)]
print("After :", df["vict_age"].min(), df["vict_age"].max())

### 2.6 Victim sex

The column has `M`, `F`, `X`, `H`, `-`, and some other one-letter codes. I collapse anything that isn't `M`/`F` into `Unknown`.

In [ ]:
df["vict_sex"] = df["vict_sex"].where(df["vict_sex"].isin(["M", "F"]), "Unknown")
df["vict_sex"].value_counts()

## 3. Crime categorization — three attempts

The dataset has 140 unique `crm_cd_desc` values. Plotting 140 bars is unreadable, so we need to bucket them into broader categories. We tried three approaches and settled on the third. Worth showing all three because the failure modes are instructive.

### 3.1 Attempt 1 — keyword matching on description

First instinct: just check whether the description string contains "theft", "burglary", "assault", etc. Looks clean, but the rule order matters and edge cases sneak through. For example, `IDENTITY THEFT` matches `theft` before it matches `identity`, so 62,000 identity-fraud rows land in the **Theft** bucket instead of **Fraud**.

In [ ]:
# Demo of why keyword matching fails:
demo = df[df["crm_cd_desc"].str.contains("IDENTITY", case=False, na=False)]
print(f"IDENTITY THEFT incidents: {len(demo):,}")
print("Would land in 'Theft' bucket if we matched on 'theft' first.")

### 3.2 Attempt 2 — `pd.cut` on the numeric code

Better idea: the LAPD code numbers *do* have structure. 100s are homicide, 200s are robbery/assault, 300s burglary, 400s theft, 500s vehicle, 600s simple assault + arson + financial, 700s vandalism + weapons, 800s sex offenses, 900s misc. So you could just bin the numeric code with `pd.cut`.

The problem is that the boundaries don't actually align. For example, codes 805, 810, 815, 820, 830 (all *sex offenses*) sit in the 800s. If you bin `(700, 900]` and label it "Vandalism & Weapons", you're silently throwing every sex offense into that bucket.

In [ ]:
# Demo of why pd.cut binning fails:
mask = (df["crm_cd"] > 700) & (df["crm_cd"] <= 900)
codes_in_that_bin = df.loc[mask, "crm_cd_desc"].value_counts().head(8)
print("What actually sits in the (700, 900] range:")
print(codes_in_that_bin)

### 3.3 Attempt 3 — explicit `{code → bucket}` dictionary  (the one we kept)

The only approach that actually respects what each code means is to read through the LAPD code list and hand-map each code to a bucket. It's tedious but unambiguous. 15 buckets total — granular enough that we can see real patterns, coarse enough to plot.

In [ ]:
CRIME_BUCKET = {
    # Homicide
    110: "Homicide", 113: "Homicide",
    # Sex offenses
    121: "Sex offenses", 122: "Sex offenses",
    760: "Sex offenses", 762: "Sex offenses",
    805: "Sex offenses", 806: "Sex offenses",
    810: "Sex offenses", 812: "Sex offenses", 813: "Sex offenses",
    814: "Sex offenses", 815: "Sex offenses",
    820: "Sex offenses", 821: "Sex offenses", 822: "Sex offenses",
    830: "Sex offenses", 840: "Sex offenses", 845: "Sex offenses",
    850: "Sex offenses", 860: "Sex offenses", 870: "Sex offenses",
    932: "Sex offenses", 956: "Sex offenses",
    # Robbery
    210: "Robbery", 220: "Robbery",
    # Aggravated assault
    230: "Aggravated assault", 231: "Aggravated assault",
    235: "Aggravated assault", 236: "Aggravated assault",
    250: "Aggravated assault", 251: "Aggravated assault",
    647: "Aggravated assault",
    # Simple assault
    622: "Simple assault", 623: "Simple assault", 624: "Simple assault",
    625: "Simple assault", 626: "Simple assault", 627: "Simple assault",
    237: "Simple assault",
    # Burglary
    310: "Burglary", 320: "Burglary", 410: "Burglary",
    # Vehicle-related
    330: "Vehicle-related", 331: "Vehicle-related",
    420: "Vehicle-related", 421: "Vehicle-related",
    433: "Vehicle-related",
    480: "Vehicle-related", 485: "Vehicle-related", 487: "Vehicle-related",
    510: "Vehicle-related", 520: "Vehicle-related", 522: "Vehicle-related",
    # Theft (other)
    341: "Theft (other)", 343: "Theft (other)", 345: "Theft (other)",
    347: "Theft (other)", 349: "Theft (other)",
    350: "Theft (other)", 351: "Theft (other)", 352: "Theft (other)", 353: "Theft (other)",
    440: "Theft (other)", 441: "Theft (other)", 442: "Theft (other)", 443: "Theft (other)",
    444: "Theft (other)", 445: "Theft (other)", 446: "Theft (other)",
    450: "Theft (other)", 451: "Theft (other)", 452: "Theft (other)", 453: "Theft (other)",
    470: "Theft (other)", 471: "Theft (other)",
    473: "Theft (other)", 474: "Theft (other)", 475: "Theft (other)",
    # Financial / fraud
    354: "Financial/fraud", 649: "Financial/fraud",
    651: "Financial/fraud", 652: "Financial/fraud", 653: "Financial/fraud", 654: "Financial/fraud",
    660: "Financial/fraud", 661: "Financial/fraud", 662: "Financial/fraud",
    664: "Financial/fraud", 666: "Financial/fraud", 668: "Financial/fraud", 670: "Financial/fraud",
    940: "Financial/fraud", 942: "Financial/fraud",
    950: "Financial/fraud", 951: "Financial/fraud",
    # Vandalism / arson
    648: "Vandalism/arson", 740: "Vandalism/arson", 745: "Vandalism/arson",
    924: "Vandalism/arson", 943: "Vandalism/arson", 949: "Vandalism/arson",
    # Weapons
    753: "Weapons", 755: "Weapons", 756: "Weapons", 761: "Weapons",
    904: "Weapons", 906: "Weapons", 931: "Weapons",
    # Threats / stalking / court order
    763: "Threats/stalking", 900: "Threats/stalking", 901: "Threats/stalking",
    902: "Threats/stalking", 903: "Threats/stalking",
    928: "Threats/stalking", 930: "Threats/stalking",
    # Public order
    432: "Public order", 434: "Public order", 435: "Public order", 436: "Public order",
    437: "Public order", 438: "Public order", 439: "Public order",
    880: "Public order", 882: "Public order", 884: "Public order",
    886: "Public order", 888: "Public order", 890: "Public order",
    933: "Public order",   # PROWLER
    # Kidnapping
    910: "Kidnapping", 920: "Kidnapping", 921: "Kidnapping", 922: "Kidnapping",
    # Other
    865: "Other", 926: "Other", 944: "Other",
    946: "Other", 948: "Other", 954: "Other",
}

df["crime_category"] = df["crm_cd"].map(CRIME_BUCKET).fillna("Other")

unique_codes = df["crm_cd"].unique()
covered = sum(1 for c in unique_codes if c in CRIME_BUCKET)
print(f"Coverage: {covered}/{len(unique_codes)} codes "
      f"({covered / len(unique_codes) * 100:.1f}%)")
print()
print(df["crime_category"].value_counts())

Good coverage. Any code we didn't map falls into `"Other"`, which is fine — those are tail codes with very few incidents each, and we don't want to over-label.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

fig, ax = plt.subplots(figsize=(10, 6))
df["crime_category"].value_counts().plot(kind="barh", ax=ax, color="steelblue")
ax.invert_yaxis()
ax.set_title("Total reported incidents by category, 2020-2024")
ax.set_xlabel("Incidents")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

Vehicle-related crimes dominate — LA is a car city, so vehicle theft + theft from vehicle + carjacking together make up the largest chunk. Theft (other) and Burglary follow. Notice that homicide, while the most serious category, is tiny by volume — about 2,500 cases in 5 years.

## 4. Analysis

### 4.1 Where do crimes happen?

LAPD divides LA into 21 geographic *areas* (Central, 77th Street, Pacific, etc.). Each area runs its own station. Total incident count by area is the obvious first question.

In [ ]:
area_counts = df["area_name"].value_counts().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
area_counts.plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Total reported incidents by LAPD area, 2020-2024")
ax.set_xlabel("Incidents")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

Central, 77th Street, and Pacific lead. That's expected: Central covers Downtown / Skid Row (very high foot traffic and homelessness concentration), 77th Street is South LA, Pacific covers Venice/LAX.

But raw count is partly a function of how big each area is. The more interesting question is **what kind of crime each area specializes in**.

In [ ]:
# Share of each crime category within each area
mix = pd.crosstab(df["area_name"], df["crime_category"], normalize="index") * 100

# Order by total incidents so the chart reads from busiest area down
mix = mix.loc[area_counts.sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(13, 9))
sns.heatmap(mix, annot=True, fmt=".1f", cmap="Blues",
            cbar_kws={"label": "% of area's incidents"}, ax=ax)
ax.set_title("Crime mix by LAPD area (% of that area's total)")
ax.set_xlabel("")
ax.set_ylabel("")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.show()

Reading rows: every area is dominated by Vehicle-related and Theft. The interesting variation is in the smaller categories — e.g. some areas have notably higher Simple Assault shares than others. We'll come back to this in the slides.

#### Density map

For the geographic view, I built a heatmap of all valid incidents. Renders inline below. (There's also a full-page version saved at `notebooks/map_la_density.html` if you want to zoom.)

In [ ]:
import plotly.express as px

# Bin to a 200x200 grid for speed
sample = df.sample(min(300_000, len(df)), random_state=0)

fig = px.density_map(
    sample,
    lat="lat", lon="lon",
    radius=3,
    center={"lat": 34.05, "lon": -118.35},
    zoom=9.2,
    map_style="open-street-map",
    height=600,
)
fig.update_layout(margin={"l": 0, "r": 0, "t": 0, "b": 0})
fig.show()

### 4.2 When do crimes happen?

Three time questions worth asking: hour-of-day, day-of-week, and month-of-year.

In [ ]:
df["dow"]   = df["date_occ"].dt.dayofweek          # 0 = Mon
df["month"] = df["date_occ"].dt.month
df["year"]  = df["date_occ"].dt.year

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

df.groupby("hour").size().plot(ax=axes[0], color="steelblue")
axes[0].set_title("By hour of day")
axes[0].set_xlabel("Hour")
axes[0].set_ylabel("Incidents")

dow_labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
df.groupby("dow").size().plot(kind="bar", ax=axes[1], color="steelblue")
axes[1].set_xticklabels(dow_labels, rotation=0)
axes[1].set_title("By day of week")
axes[1].set_xlabel("")

df.groupby("month").size().plot(kind="bar", ax=axes[2], color="steelblue")
axes[2].set_title("By month")
axes[2].set_xlabel("Month")

plt.tight_layout()
plt.show()

Two things to call out:
- **The hour chart has a spike at hour 12.** That's not a real lunchtime spike — it's the convention that an incident with no recorded time gets logged as `1200`. We'll flag this as a data quality issue in Limitations.
- The Friday peak in weekly counts and the late-summer (July/August) peak in monthly counts are real and consistent with the criminology literature (more outdoor activity, longer days, schools out).

#### Hour patterns by category

Different crime types peak at different times.

In [ ]:
top_cats = df["crime_category"].value_counts().head(8).index.tolist()
hour_by_cat = (
    df[df["crime_category"].isin(top_cats)]
    .groupby(["hour", "crime_category"]).size()
    .unstack()
    .reindex(columns=top_cats)
)

fig, ax = plt.subplots(figsize=(13, 6))
hour_by_cat.plot(ax=ax, lw=2)
ax.set_title("Incidents by hour and category (top 8 categories)")
ax.set_xlabel("Hour")
ax.set_ylabel("Incidents")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="Category")
plt.tight_layout()
plt.show()

Theft and Vehicle-related crimes peak in the late afternoon/evening — they're driven by opportunity (cars left in lots, retail open). Aggravated and Simple Assault peak later, around evening and into the night, consistent with alcohol involvement.

### 4.3 Did COVID change crime?

LA's stay-at-home order ran roughly March 19, 2020 → June 15, 2021. A clean before/after design would need pre-2020 data, which this dataset doesn't have. So I treat the question as an *interrupted time series within the dataset* — looking at the trajectory of monthly counts and category mix from 2020 to 2024.

In [ ]:
monthly = df.groupby(pd.Grouper(key="date_occ", freq="MS")).size()

fig, ax = plt.subplots(figsize=(13, 5))
monthly.plot(ax=ax, color="steelblue")
ax.axvspan("2020-03-19", "2021-06-15", color="orange", alpha=0.18,
           label="LA stay-at-home order")
ax.set_title("Total reported incidents per month, 2020-2024")
ax.set_xlabel("")
ax.set_ylabel("Incidents")
ax.legend()
plt.tight_layout()
plt.show()

The dip in early 2020 is real and matches the lockdown. The recovery is clear by mid-2021. The decline from late 2022 onward is also real and consistent with LAPD's reporting of falling crime in their 2023/2024 statistics.

Now the same story by category share — did the *mix* shift?

In [ ]:
cat_share = (
    df.groupby([pd.Grouper(key="date_occ", freq="QS"), "crime_category"])
      .size()
      .unstack(fill_value=0)
)
cat_share_pct = cat_share.div(cat_share.sum(axis=1), axis=0) * 100

# Plot top-6 categories to keep readable
top6 = df["crime_category"].value_counts().head(6).index.tolist()

fig, ax = plt.subplots(figsize=(13, 6))
cat_share_pct[top6].plot.area(ax=ax, alpha=0.7)
ax.set_title("Crime category share over time (top 6, quarterly)")
ax.set_xlabel("")
ax.set_ylabel("Share of all incidents (%)")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

Vehicle-related crime grew from ~25% pre-pandemic to ~30% post-pandemic, mostly at the expense of Theft (other) — consistent with shoplifting being suppressed during lockdown closures and vehicle theft becoming relatively more attractive (parked cars sitting unused).

### 4.4 Does temperature drive crime?

This is the angle Arminas added — a creative idea. Criminology literature suggests warm weather increases interpersonal violence (more time outdoors, more contact). To test it in this dataset, I pull hourly temperature for LA from Open-Meteo's free archive API, merge it onto every incident, and check whether warmer hours have more crime.

**Important caveat upfront:** raw `crime_count ~ temp` is misleading because both crime and temperature peak in the same hours (3-5 PM), creating a spurious correlation. So we'll plot both unconditional and **controlled for hour of day**.

In [ ]:
import requests

# Open-Meteo archive — free, no API key. We're asking for hourly temp
# at central LA across our whole study window.
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 34.05,
    "longitude": -118.24,
    "start_date": "2020-01-01",
    "end_date": "2024-12-30",
    "hourly": "temperature_2m",
    "timezone": "America/Los_Angeles",
}
resp = requests.get(url, params=params, timeout=60)
resp.raise_for_status()
payload = resp.json()

weather = pd.DataFrame({
    "ts": pd.to_datetime(payload["hourly"]["time"]),
    "temp_c": payload["hourly"]["temperature_2m"],
})
weather["date"] = weather["ts"].dt.date
weather["hour"] = weather["ts"].dt.hour
print(f"Weather rows: {len(weather):,}, "
      f"range {weather['ts'].min()} → {weather['ts'].max()}")

In [ ]:
# Merge temp onto every crime
df["_date"] = df["date_occ"].dt.date
crime_w = df.merge(
    weather[["date", "hour", "temp_c"]],
    left_on=["_date", "hour"],
    right_on=["date", "hour"],
    how="left",
).drop(columns=["_date", "date"])

print(f"With temp: {crime_w['temp_c'].notna().sum():,} / {len(crime_w):,}")

#### Unconditional view (the misleading one)

In [ ]:
crime_w["temp_bin"] = pd.cut(crime_w["temp_c"], bins=range(0, 45, 2))
unconditional = crime_w.groupby("temp_bin", observed=True).size()

fig, ax = plt.subplots(figsize=(10, 5))
unconditional.plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Incidents by temperature bin (unconditional — misleading)")
ax.set_xlabel("Temperature (°C, 2°C bins)")
ax.set_ylabel("Incidents")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Looks like crime peaks around 18-24°C — but that's just the range where most LA hours sit. To see whether temp has an effect *at the same time of day*, we need to control for hour.

#### Controlled view: rate per hour-of-day-of-temperature

In [ ]:
# Count incidents per (hour, temp_bin) and divide by how many hours
# of clock-time the city actually spent at that temp.
hours_at_temp = (
    weather.assign(temp_bin=pd.cut(weather["temp_c"], bins=range(0, 45, 2)))
           .groupby(["hour", "temp_bin"], observed=True)
           .size()
           .rename("hours_observed")
)

incidents_at_temp = (
    crime_w.groupby(["hour", "temp_bin"], observed=True)
           .size()
           .rename("incidents")
)

rate = pd.concat([incidents_at_temp, hours_at_temp], axis=1).dropna()
rate["per_hour"] = rate["incidents"] / rate["hours_observed"]

# Average rate across hours, weighted by hours observed
avg_rate_by_temp = (
    rate.groupby("temp_bin", observed=True)
        .apply(lambda g: (g["incidents"].sum() / g["hours_observed"].sum()),
               include_groups=False)
)

fig, ax = plt.subplots(figsize=(10, 5))
avg_rate_by_temp.plot(kind="bar", ax=ax, color="darkorange")
ax.set_title("Incidents per clock-hour at each temperature (controlled)")
ax.set_xlabel("Temperature (°C, 2°C bins)")
ax.set_ylabel("Incidents per hour spent at that temp")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Now the spurious peak flattens, and we can see whether temperature has any residual effect after accounting for the fact that hot hours are also busy hours. **The signal is much weaker than the unconditional view suggested** — most of the apparent "heat → crime" pattern was actually just "afternoons are busy".

There is still a mild upward trend in the warmer bins, which is consistent with the criminology literature on temperature and violence, but it's a fraction of the size you'd guess from the uncontrolled chart. Good lesson in why uncontrolled correlations mislead.

## 5. Limitations

Things we'd flag for the Q&A and slide on limitations:

1. **This is reported crime, not actual crime.** Categories with low reporting rates (sexual assault, domestic violence) are systematically undercounted. Comparing area-to-area is partly comparing how much each community reports.

2. **No pre-2020 baseline.** Our COVID story is descriptive (within-window trajectory) not causal. To do a real impact study we'd want at least 2017-2019.

3. **The hour-12 artifact.** Incidents with no recorded time default to `1200`, so the "noon spike" in the hour-of-day chart is partly data quality. A more careful analysis would either drop these or distribute them.

4. **NIBRS migration in March 2024.** Categories started to be coded differently and LAPD froze this dataset. The last few months of 2024 may be artificially low.

5. **Geocoding precision.** LAPD intentionally jitters lat/lon to the nearest 100 block for victim privacy. Heatmaps are accurate to neighborhood scale but not block-level.

6. **Bucketing is a judgment call.** Our 15 buckets are defensible but other reasonable choices exist (e.g. UCR Part 1 vs Part 2). We'd want to do a sensitivity check before publishing any finding that depends on the exact grouping.

7. **Things we wish we had:** suspect demographics (currently we only have victim), case outcome details (we have status but not court disposition), and a pre-2020 baseline.